# RFD Urban Digital Twin Analysis

This notebook builds compact exam pipeline for RFD-based consistency profiling on urban air-quality data.

Scope:
- simplified conceptual Urban Digital Twin;
- Beijing air-quality subset with two stations;
- interpretable approximate rules, not causal claims;
- lightweight discovery over reduced deterministic sample.


## Dataset Configuration

Selected subset:
- stations: `Aotizhongxin`, `Changping`
- period: `2013-09-01` to `2013-12-31`
- cleaned dataset target: around 5k-6k rows
- RFD experiment sample: 1500 rows balanced across stations

Why December included: Sep-Nov cleaned subset produced only 3996 rows after dropping missing values. Adding December yields 5426 rows.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

current = Path.cwd().resolve()
PROJECT_ROOT = next(path for path in [current, *current.parents] if (path / 'AGENTS.md').exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = PROJECT_ROOT / 'figures'

for path in [DATA_PROCESSED, RESULTS_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)


In [ ]:
from src.experiments import (
    prepare_rfd_sample,
    run_candidate_validation,
    run_lightweight_discovery,
    run_station_comparison,
    run_threshold_comparison,
    select_top_rule_violations,
    export_violation_examples,
)
from src.preprocessing import prepare_udt_dataset
from src.profiling import (
    correlation_matrix,
    data_types_summary,
    dataset_shape_summary,
    hour_distribution,
    missing_values_summary,
    numeric_summary,
    station_distribution,
    time_slot_distribution,
    unique_values_summary,
)
from src.visualization import (
    plot_correlation_matrix,
    plot_missing_values,
    plot_pm25_timeseries,
    plot_pollutant_distribution,
)


## Step 1-2: Preprocessing

In [ ]:
cleaned_df, pre_drop_missing = prepare_udt_dataset(
    raw_dir=DATA_RAW,
    processed_path=DATA_PROCESSED / 'udt_rfd_dataset.csv',
)
sample_df = prepare_rfd_sample(cleaned_df)
sample_df.to_csv(DATA_PROCESSED / 'udt_rfd_sample.csv', index=False)

print('Cleaned rows:', len(cleaned_df))
print('Cleaned columns:', cleaned_df.shape[1])
print('RFD sample rows:', len(sample_df))
display(pre_drop_missing)


## Step 3: Profiling

In [ ]:
shape_df = dataset_shape_summary(cleaned_df)
dtypes_df = data_types_summary(cleaned_df)
missing_df = missing_values_summary(cleaned_df)
unique_df = unique_values_summary(cleaned_df)
numeric_df = numeric_summary(cleaned_df)
station_df = station_distribution(cleaned_df)
hour_df = hour_distribution(cleaned_df)
time_slot_df = time_slot_distribution(cleaned_df)
corr_df = correlation_matrix(
    cleaned_df,
    numeric_columns=['hour', 'PM2.5', 'PM10', 'NO2', 'O3', 'TEMP', 'DEWP', 'WSPM'],
)

shape_df.to_csv(RESULTS_DIR / 'profile_shape_summary.csv', index=False)
dtypes_df.to_csv(RESULTS_DIR / 'profile_dtypes_summary.csv', index=False)
missing_df.to_csv(RESULTS_DIR / 'profile_missing_summary.csv', index=False)
unique_df.to_csv(RESULTS_DIR / 'profile_unique_values_summary.csv', index=False)
numeric_df.to_csv(RESULTS_DIR / 'profile_numeric_summary.csv', index=False)
station_df.to_csv(RESULTS_DIR / 'profile_station_distribution.csv', index=False)
hour_df.to_csv(RESULTS_DIR / 'profile_hour_distribution.csv', index=False)
time_slot_df.to_csv(RESULTS_DIR / 'profile_time_slot_distribution.csv', index=False)
corr_df.to_csv(RESULTS_DIR / 'profile_correlation_matrix.csv')

plot_missing_values(missing_df, FIGURES_DIR / 'missing_values.png')
plot_pollutant_distribution(cleaned_df, 'PM2.5', FIGURES_DIR / 'pm25_distribution.png')
plot_pollutant_distribution(cleaned_df, 'NO2', FIGURES_DIR / 'no2_distribution.png')
plot_pm25_timeseries(cleaned_df, FIGURES_DIR / 'pm25_timeseries.png')
plot_correlation_matrix(corr_df, FIGURES_DIR / 'correlation_matrix.png')

print('Dataset shape')
display(shape_df)
print('Data types')
display(dtypes_df)
print('Missing values after cleaning')
display(missing_df)
print('Unique values')
display(unique_df)
print('Numeric summary')
display(numeric_df)
print('Station distribution')
display(station_df)
print('Hour distribution')
display(hour_df)
print('Time-slot distribution')
display(time_slot_df)


## Step 4-6: Candidate RFD Validation

Threshold sets are defined in `src/rfd.py` as `strict`, `medium`, and `relaxed`.
Candidate rules are evaluated on balanced 1500-row sample to keep pairwise validation tractable.


In [ ]:
candidate_df, candidate_violations = run_candidate_validation(
    sample_df,
    RESULTS_DIR / 'rfd_candidate_results.csv',
)

candidate_table = candidate_df.loc[:, ['rule_label', 'support', 'confidence', 'violations', 'interpretation']].copy()
for column in ['support', 'confidence']:
    candidate_table[column] = candidate_table[column].round(4)

display(candidate_table)


## Step 7: Lightweight Discovery

In [ ]:
discovery_df = run_lightweight_discovery(
    sample_df,
    RESULTS_DIR / 'rfd_discovered_top10.csv',
)

if discovery_df.empty:
    print('No discovered RFD met support >= 0.01 and confidence >= 0.85 under medium thresholds.')
else:
    display(discovery_df.round(4))


## Step 8-9: Threshold and Station Comparisons

In [ ]:
threshold_df = run_threshold_comparison(
    sample_df,
    RESULTS_DIR / 'rfd_threshold_comparison.csv',
    FIGURES_DIR / 'rfd_confidence_by_threshold.png',
)
station_df_rfd = run_station_comparison(
    sample_df,
    RESULTS_DIR / 'rfd_station_comparison.csv',
    FIGURES_DIR / 'rfd_confidence_by_station.png',
)

display(threshold_df[['rule_label', 'threshold_set', 'support', 'confidence', 'violations']].round(4))
display(station_df_rfd[['rule_label', 'station_scope', 'support', 'confidence', 'violations']].round(4))


## Step 10: Violation Analysis

In [ ]:
selected_violations = select_top_rule_violations(candidate_df, candidate_violations, top_rules=2, per_rule=5)
violations_df = export_violation_examples(
    selected_violations,
    RESULTS_DIR / 'violations_examples.csv',
    limit=10,
)

display(violations_df)


## Takeaways

- Candidate RFDs are interpretable but moderately weak on this subset; none behaves like near-deterministic rule.
- Relaxed thresholds increase confidence and support, but still do not yield discovery results above project filter (`0.01`, `0.85`).
- Violating pairs remain useful for UDT framing: they can indicate local inconsistency, unusual pollution dynamics, or missing contextual variables.
- This is approximate consistency analysis, not prediction and not causality.
